In [1]:
# ==============================================================================
# CELL 1: Package Installations
# ==============================================================================
# omnivoice requires transformers>=5.3.0, so everything is installed here to
# give every later cell (ASR, LLM, TTS) the same transformers version.
#
# If an older transformers was already imported in this kernel, use
# Runtime > Restart session before re-running from the top.
#
# peft/bitsandbytes/torchao are not needed by the fp16 LLM cell; they are kept
# for optional LoRA fine-tuning of CohereLabs/tiny-aya-base.
!pip install -q -U "transformers>=5.3.0" torchaudio librosa accelerate soundfile jiwer
!pip install -q -U peft bitsandbytes torchao
!pip install -q omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ==============================================================================
# CELL 2: System Verification & Device Setup
# ==============================================================================
import re
import torch
from IPython.display import HTML, Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Active Execution Device: {device}")
if device == "cuda":
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Allocated: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✅ Active Execution Device: cuda
   GPU Model: Tesla T4
   VRAM Allocated: 15.64 GB


In [3]:
# ==============================================================================
# CELL 3: Fidel Text Normalization Engine
# ==============================================================================
def normalize_amharic_text(text: str) -> str:
    """Standardizes Amharic text by folding Ge'ez homophones and removing punctuation."""
    if not text:
        return ""

    text = re.sub(r'\[AMH\]', '', text)

    # Homophone folding
    text = re.sub(r'[ሐኀኻⶠ]', 'ሀ', text)
    text = re.sub(r'[ሑኁኹ]', 'ሁ', text)
    text = re.sub(r'[ሒኂኺ]', 'ሂ', text)
    text = re.sub(r'[ሓኃኻ]', 'ሃ', text)
    text = re.sub(r'[ሔኄኼ]', 'ሄ', text)
    text = re.sub(r'[ሕኅኽ]', 'ህ', text)
    text = re.sub(r'[ሖኆኾ]', 'ሆ', text)

    text = re.sub(r'[ሠ]', 'ሰ', text)
    text = re.sub(r'[ሡ]', 'ሱ', text)
    text = re.sub(r'[ሢ]', 'ሲ', text)
    text = re.sub(r'[ሣ]', 'ሳ', text)
    text = re.sub(r'[ሤ]', 'ሴ', text)
    text = re.sub(r'[ሥ]', 'ስ', text)
    text = re.sub(r'[ሦ]', 'ሶ', text)

    text = re.sub(r'[ዐ]', 'አ', text)
    text = re.sub(r'[ዑ]', 'ኡ', text)
    text = re.sub(r'[ዒ]', 'ኢ', text)
    text = re.sub(r'[ዓ]', 'ኣ', text)
    text = re.sub(r'[ዔ]', 'ኤ', text)
    text = re.sub(r'[ዕ]', 'እ', text)
    text = re.sub(r'[ዖ]', 'ኦ', text)

    text = re.sub(r'[ፀ]', 'ጸ', text)
    text = re.sub(r'[ፁ]', 'ጹ', text)
    text = re.sub(r'[ፂ]', 'ጺ', text)
    text = re.sub(r'[ፃ]', 'ጻ', text)
    text = re.sub(r'[ፄ]', 'ጼ', text)
    text = re.sub(r'[ፅ]', 'ጽ', text)
    text = re.sub(r'[ፆ]', 'ጾ', text)

    text = re.sub(r'[\u1360-\u1368\?\!\.\,\-\"\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("✅ Normalization Engine Ready.")

✅ Normalization Engine Ready.


In [4]:
# ==============================================================================
# CELL 4: Browser Microphone Audio Recorder Widget (HTML5/JS with Native WAV)
# ==============================================================================
RECORD_HTML = """
<div style="border: 2px solid #4CAF50; padding: 15px; border-radius: 10px; width: 360px; text-align: center; font-family: sans-serif;">
    <h4>🎙️ Amharic Voice Assistant</h4>
    <button id="startBtn" style="background-color: #4CAF50; color: white; padding: 10px 20px; border: none; border-radius: 5px; cursor: pointer; margin-right: 10px;">Record</button>
    <button id="stopBtn" style="background-color: #f44336; color: white; padding: 10px 20px; border: none; border-radius: 5px; cursor: pointer;" disabled>Stop</button>
    <p id="status" style="margin-top: 10px; color: #555; font-size: 14px;">Status: Ready</p>
</div>

<script>
let audioContext;
let mediaStream;
let recProcessor;
let audioBuffers = [];

function encodeWAV(samples, sampleRate) {
    let buffer = new ArrayBuffer(44 + samples.length * 2);
    let view = new DataView(buffer);

    /* RIFF identifier */
    for (let i = 0; i < 4; i++) view.setUint8(i, "RIFF".charCodeAt(i));
    /* RIFF chunk length */
    view.setUint32(4, 36 + samples.length * 2, true);
    /* RIFF type */
    for (let i = 0; i < 4; i++) view.setUint8(8 + i, "WAVE".charCodeAt(i));
    /* format chunk identifier */
    for (let i = 0; i < 4; i++) view.setUint8(12 + i, "fmt ".charCodeAt(i));
    /* format chunk length */
    view.setUint32(16, 16, true);
    /* sample format (PCM) */
    view.setUint16(20, 1, true);
    /* channel count (mono) */
    view.setUint16(22, 1, true);
    /* sample rate */
    view.setUint32(24, sampleRate, true);
    /* byte rate */
    view.setUint32(28, sampleRate * 2, true);
    /* block align */
    view.setUint16(32, 2, true);
    /* bits per sample */
    view.setUint16(34, 16, true);
    /* data chunk identifier */
    for (let i = 0; i < 4; i++) view.setUint8(36 + i, "data".charCodeAt(i));
    /* data chunk length */
    view.setUint32(40, samples.length * 2, true);

    /* PCM sample encoding */
    let index = 44;
    for (let i = 0; i < samples.length; i++) {
        let s = Math.max(-1, Math.min(1, samples[i]));
        view.setInt16(index, s < 0 ? s * 0x8000 : s * 0x7FFF, true);
        index += 2;
    }
    return new Blob([view], { type: 'audio/wav' });
}

document.getElementById('startBtn').onclick = async () => {
    try {
        mediaStream = await navigator.mediaDevices.getUserMedia({ audio: true });
        audioContext = new (window.AudioContext || window.webkitAudioContext)({ sampleRate: 16000 });
        let source = audioContext.createMediaStreamSource(mediaStream);

        recProcessor = audioContext.createScriptProcessor(4096, 1, 1);
        audioBuffers = [];

        recProcessor.onaudioprocess = (e) => {
            let inputBuffer = e.inputBuffer.getChannelData(0);
            audioBuffers.push(new Float32Array(inputBuffer));
        };

        source.connect(recProcessor);
        recProcessor.connect(audioContext.destination);

        document.getElementById('startBtn').disabled = true;
        document.getElementById('stopBtn').disabled = false;
        document.getElementById('status').innerText = "🔴 Recording... Speak in Amharic";
    } catch (err) {
        document.getElementById('status').innerText = "❌ Mic Error: " + err.message;
    }
};

document.getElementById('stopBtn').onclick = async () => {
    try {
        recProcessor.disconnect();
        mediaStream.getTracks().forEach(track => track.stop());

        let totalLength = audioBuffers.reduce((acc, buf) => acc + buf.length, 0);
        let pcmData = new Float32Array(totalLength);
        let offset = 0;
        for (let buf of audioBuffers) {
            pcmData.set(buf, offset);
            offset += buf.length;
        }

        let wavBlob = encodeWAV(pcmData, 16000);
        let reader = new FileReader();
        reader.readAsDataURL(wavBlob);
        reader.onloadend = () => {
            let base64Audio = reader.result.split(',')[1];
            document.getElementById('status').innerText = "⏳ Processing in Python...";

            google.colab.kernel.invokeFunction('notebook.process_audio', [base64Audio], {})
                .then(() => {
                    document.getElementById('status').innerText = "✅ Completed! Ready for next input.";
                })
                .catch(err => {
                    document.getElementById('status').innerText = "❌ Python Error (See cell output below)";
                    console.error("Colab Kernel Error:", err);
                });
        };
    } catch (err) {
        document.getElementById('status').innerText = "❌ Encoding Error: " + err.message;
    }

    document.getElementById('startBtn').disabled = false;
    document.getElementById('stopBtn').disabled = true;
};
</script>
"""

def record_microphone_audio():
    """Renders the HTML widget."""
    display(HTML(RECORD_HTML))

In [5]:
# ==============================================================================
# CELL 5: Load ASR Model (snapwre/hohe-asr-amharic)
# ==============================================================================
import gc
import torch
import soundfile as sf
from transformers import AutoProcessor, AutoModelForCTC

# Free up any fragmented CUDA memory before loading
gc.collect()
torch.cuda.empty_cache()

asr_model_id = "snapwre/hohe-asr-amharic"
device = "cuda" if torch.cuda.is_available() else "cpu"

asr_processor = AutoProcessor.from_pretrained(asr_model_id)
asr_model = AutoModelForCTC.from_pretrained(asr_model_id).to(device)
asr_model.eval()
print(f"✅ ASR Model Loaded on {device}.")


def transcribe_amharic_audio(audio_path):
    # Load audio (ensure 16kHz for wav2vec2-bert)
    audio_input, sample_rate = sf.read(audio_path)

    inputs = asr_processor(audio_input, sampling_rate=sample_rate, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = asr_model(**inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = asr_processor.batch_decode(predicted_ids)[0]

    # Strip the [AMH] tag generated by hohe-asr-amharic
    transcription = transcription.replace("[AMH]", "").strip()

    return transcription

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/32.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/797 [00:00<?, ?it/s]

✅ ASR Model Loaded on cuda.


### CELL 6: Load LLM Engine — Cohere Labs Tiny Aya Earth (3.35B, fp16)

**Gated model — one-time setup required:**
1. Visit https://huggingface.co/CohereLabs/tiny-aya-earth and accept the license (CC-BY-NC-4.0, non-commercial use only, plus Cohere's Acceptable Use Policy).
2. Create an HF access token: https://huggingface.co/settings/tokens
3. In Colab, add it as a secret named `HF_TOKEN` (key icon, left sidebar), or set the `HF_TOKEN` environment variable before running the next cell.

In [6]:
import gc
import os
import re

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Clear VRAM before starting
gc.collect()
torch.cuda.empty_cache()

# --- Authenticate (required: Tiny Aya is a gated repo) ---
_hf_token = os.environ.get("HF_TOKEN")
if _hf_token is None:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        print("loaded token")
    except Exception:
        _hf_token = None

if _hf_token:
    from huggingface_hub import login as _hf_login
    _hf_login(token=_hf_token)
else:
    print("⚠️ No HF_TOKEN found. CohereLabs/tiny-aya-earth is gated — "
          "accept the license at https://huggingface.co/CohereLabs/tiny-aya-earth "
          "and add an HF_TOKEN secret, or this cell will fail to download.")

print("⏳ Loading Tiny Aya Earth (CohereLabs/tiny-aya-earth) in fp16...")

LLM_MODEL_ID = "CohereLabs/tiny-aya-earth"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, token=_hf_token)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    token=_hf_token,
)
llm_model.eval()

print(f"✅ Tiny Aya Earth Loaded Successfully on {device} (fp16).")

def sanitize_to_ethiopic(text: str) -> str:
    """Strips foreign token hallucinations and keeps Amharic script."""
    ethiopic_only = re.sub(r'[^\u1200-\u137F\s0-9\.\,\?\!]', '', text)
    return re.sub(r'\s+', ' ', ethiopic_only).strip()

# Keeps replies short and spoken-style, which matters once the text is
# piped into TTS and spoken aloud.
_SYSTEM_PROMPT = (
    "You are a helpful voice assistant that always replies in Amharic. "
    "Keep natural, and conversational, as if speaking out "
    "loud — at most 2-3 sentences unless the user explicitly asks for more detail."
)

def generate_llm_response(prompt_text: str) -> str:
    """Generates a short, spoken-style Amharic response using Tiny Aya Earth."""

    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device)
    except Exception:
        # Fall back to a user-only turn if the template rejects a system role.
        inputs = llm_tokenizer.apply_chat_template(
            [{"role": "user", "content": f"{_SYSTEM_PROMPT}\n\n{prompt_text}"}],
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device)

    with torch.inference_mode():
        generated_ids = llm_model.generate(
            **inputs,
            max_new_tokens=512,   # capped for latency
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=llm_tokenizer.pad_token_id or llm_tokenizer.eos_token_id,
        )

    new_tokens = generated_ids[0][inputs["input_ids"].shape[-1]:]
    raw_response = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    clean_response = sanitize_to_ethiopic(raw_response)

    if not clean_response:
        clean_response = "ሰላም! እንዴት ልረዳህ እችላለሁ?"

    return clean_response

loaded token
⏳ Loading Tiny Aya Earth (CohereLabs/tiny-aya-earth) in fp16...


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Tiny Aya Earth Loaded Successfully on cuda (fp16).


In [7]:
# ==============================================================================
# CELL 7: High-Quality Amharic Text-to-Speech Engine (via OmniVoice)
# ==============================================================================
import gc
import numpy as np
import torch
from omnivoice import OmniVoice, OmniVoiceGenerationConfig

OMNIVOICE_MODEL_ID = "african-low-resource/omnivoice-amharic"

# Free VRAM before loading; OmniVoice has to fit alongside the ASR and LLM.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_tts_device = "cuda:0" if device == "cuda" else "cpu"
_tts_dtype = torch.float16 if _tts_device.startswith("cuda") else torch.float32

def _load_omnivoice(target_device, target_dtype):
    print(f"⏳ Loading OmniVoice ({OMNIVOICE_MODEL_ID}) on {target_device}...")
    return OmniVoice.from_pretrained(
        OMNIVOICE_MODEL_ID,
        device_map=target_device,
        dtype=target_dtype,
    )

try:
    omnivoice_model = _load_omnivoice(_tts_device, _tts_dtype)
except RuntimeError as e:
    if "out of memory" not in str(e).lower():
        raise
    print("⚠️ Not enough VRAM to share the GPU with the LLM — retrying OmniVoice on CPU.")
    _tts_device, _tts_dtype = "cpu", torch.float32
    gc.collect()
    torch.cuda.empty_cache()
    omnivoice_model = _load_omnivoice(_tts_device, _tts_dtype)

omnivoice_model.eval()
TTS_SAMPLE_RATE = omnivoice_model.sampling_rate or 24000
print(f"✅ OmniVoice Loaded on {_tts_device} (sampling rate: {TTS_SAMPLE_RATE} Hz).")

# Auto-voice mode picks a voice on its own; a fixed seed keeps it consistent
# across turns. Remove the torch.manual_seed(...) call to vary it each time.
_TTS_SEED = 42

def synthesize_speech(text: str):
    """Converts Amharic text into speech using OmniVoice (auto voice mode)."""
    if not text or not text.strip():
        print("⚠️ Empty text provided to TTS.")
        return np.zeros(TTS_SAMPLE_RATE), TTS_SAMPLE_RATE

    print(f"🔤 TTS Input (Raw): {text}")

    try:
        torch.manual_seed(_TTS_SEED)
        with torch.inference_mode():
            audio_list = omnivoice_model.generate(
                text=text,
                language="Amharic",
                generation_config=OmniVoiceGenerationConfig(
                    num_step=32,
                    guidance_scale=2.0,
                ),
            )
        audio_data = audio_list[0]
        if isinstance(audio_data, torch.Tensor):
            audio_data = audio_data.detach().cpu().float().numpy()
        audio_data = np.asarray(audio_data).squeeze()
        return audio_data, TTS_SAMPLE_RATE
    except Exception as e:
        print(f"⚠️ TTS generation error ({e}).")
        return np.zeros(TTS_SAMPLE_RATE), TTS_SAMPLE_RATE

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


⏳ Loading OmniVoice (african-low-resource/omnivoice-amharic) on cuda:0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

✅ OmniVoice Loaded on cuda:0 (sampling rate: 24000 Hz).


In [8]:
# ==============================================================================
# CELL 8: Audio Pipeline Orchestrator & Colab Callback Registration
# ==============================================================================
from google.colab import output
import base64
import tempfile

def process_audio(base64_audio_str: str):
    """Backend processor callback triggered when the user stops recording."""
    try:
        print("\n--- New Voice Interaction ---")

        # 1. Decode base64 audio payload
        audio_bytes = base64.b64decode(base64_audio_str)

        # 2. Save the incoming audio bytes to a temporary WAV file
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temp_wav:
            temp_wav.write(audio_bytes)
            temp_wav_path = temp_wav.name

        # 3. Transcribe speech using high-level transcription function from Cell 5
        print("🎤 Transcribing audio...")
        user_query = transcribe_amharic_audio(temp_wav_path)
        print(f"🗣️ User (Transcribed): {user_query}")

        if not user_query.strip():
            user_query = "ሰላም"
            print("⚠️ Transcription was empty, defaulting query to 'ሰላም'")

        # 4. Generate response using Cohere Labs Tiny Aya Earth
        print("🧠 Generating AI response...")
        ai_response = generate_llm_response(user_query)
        print(f"🤖 Assistant: {ai_response}")

        # 5. Synthesize TTS speech using african-low-resource/omnivoice-amharic
        print("🔊 Generating audio speech output...")
        speech_wave, tts_sr = synthesize_speech(ai_response)

        # 6. Auto-play the response in the notebook output
        display(Audio(speech_wave, rate=tts_sr, autoplay=True))

    except Exception as e:
        print(f"❌ Error in audio processing pipeline: {e}")

# Register the Python function as a callback accessible by JavaScript
output.register_callback('notebook.process_audio', process_audio)
print("✅ Orchestration Pipeline & Colab Callback Registered.")

✅ Orchestration Pipeline & Colab Callback Registered.


In [9]:
# ==============================================================================
# CELL 9: Run Voice Assistant Session
# ==============================================================================
# Render the recorder widget. Clicking 'Stop' automatically triggers Python processing.
record_microphone_audio()

Output hidden; open in https://colab.research.google.com to view.